# 🚀 DWT-MobileUNet: Huấn luyện, Lượng tử hóa INT8 & Demo trên Google Colab
**Đề tài:** *Design of an Embedded System for Image Forgery Detection and Localization using Multiresolution Analysis and Deep Learning*

**Quy trình thực thi tự động trên Google Colab (T4 / L4 / A100):**
1. Kiểm tra GPU CUDA & Mount Google Drive (lưu trữ bền vững Dataset + Checkpoint `best_model.pth`).
2. Clone hoặc `git pull` mã nguồn từ GitHub.
3. Cài đặt thư viện phụ thuộc (`requirements_colab.txt`).
4. Chạy kiểm thử nhanh `--dry-run` (3 giây) để xác minh kiến trúc 2-Level Haar DWT + MobileUNet Dual-Head và CUDA AMP.
5. Chuẩn bị tập dữ liệu (CASIA v2.0 / Columbia / NIST16) và chạy huấn luyện chính thức qua CLI `train.py`.
6. Xuất mô hình sang ONNX & lượng tử hóa INT8 (`< 2 MB`) để triển khai lên **Raspberry Pi 3 Model B**.
7. Khởi chạy giao diện trực quan **Gradio** (`share=True`) ngay trên Colab.

### Bước 1: Kiểm tra GPU & Mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')

# Thư mục lưu trữ lâu dài trên Google Drive (không bị mất khi Colab ngắt phiên)
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DATN_DWT_MobileUNet'
DRIVE_CKPT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'
DRIVE_DATA_DIR = f'{DRIVE_PROJECT_DIR}/dataset/CASIA2'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'✅ Thư mục Checkpoint trên Google Drive: {DRIVE_CKPT_DIR}')

### Bước 2: Đồng bộ mã nguồn từ GitHub (Clone / Pull)

In [ ]:
# Thay URL_GITHUB_REPO bằng đường dẫn repository của bạn
GITHUB_REPO_URL = 'https://github.com/<username>/<repo-name>.git'
REPO_DIR = '/content/DATN_VER2'

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!ls -la

### Bước 3: Cài đặt thư viện (`requirements_colab.txt`)

In [ ]:
!pip install -q -r requirements_colab.txt
print('✅ Đã cài đặt đầy đủ thư viện trên Google Colab!')

### Bước 4: Kiểm thử nhanh kiến trúc & CUDA AMP (`--dry-run` trong 3 giây)
Chạy 1 bước *Forward $\rightarrow$ Multi-Task Loss $\rightarrow$ Backward $\rightarrow$ Optimizer* với synthetic tensor để xác nhận toàn bộ shape $(B, 9, 192, 192)$, số lượng tham số và GPU hoạt động trơn tru trước khi huấn luyện dài.

In [ ]:
!python train.py --dry-run --save_dir {DRIVE_CKPT_DIR} --amp

### Bước 5: (Tùy chọn) Tạo tập dữ liệu mẫu hoặc copy Dataset từ Drive sang ổ cứng máy ảo Colab
> **Mẹo tăng tốc I/O trên Colab:** Copy dữ liệu từ `/content/drive/MyDrive/...` sang `/content/dataset` giúp tốc độ đọc ảnh nhanh hơn 8–10 lần trong suốt quá trình train.

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

LOCAL_DATA_DIR = '/content/dataset/CASIA2'

if os.path.exists(DRIVE_DATA_DIR):
    print(f'📂 Đang đồng bộ dataset từ Google Drive ({DRIVE_DATA_DIR}) sang {LOCAL_DATA_DIR}...')
    os.makedirs('/content/dataset', exist_ok=True)
    !cp -rn {DRIVE_DATA_DIR} /content/dataset/
    print('✅ Đồng bộ hoàn tất!')
else:
    print('⚠️ Chưa tìm thấy dataset trên Drive. Khởi tạo bộ dữ liệu mẫu (Au / Tp / Gt_mask) để kiểm thử pipeline...')
    for sub in ['Au', 'Tp', 'Gt_mask']:
        Path(f'{LOCAL_DATA_DIR}/{sub}').mkdir(parents=True, exist_ok=True)
    for i in range(16):
        au_img = np.random.randint(0, 256, (256, 384, 3), dtype=np.uint8)
        Image.fromarray(au_img).save(f'{LOCAL_DATA_DIR}/Au/Au_sample_{i:03d}.jpg', quality=92)
        tp_img = np.random.randint(0, 256, (256, 384, 3), dtype=np.uint8)
        gt_mask = np.zeros((256, 384), dtype=np.uint8)
        gt_mask[64:160, 96:220] = 255
        tp_img[64:160, 96:220] = np.clip(tp_img[64:160, 96:220] + 45, 0, 255)
        Image.fromarray(tp_img).save(f'{LOCAL_DATA_DIR}/Tp/Tp_sample_{i:03d}.jpg', quality=75)
        Image.fromarray(gt_mask).save(f'{LOCAL_DATA_DIR}/Gt_mask/Tp_sample_{i:03d}_gt.png')
    print(f'✅ Đã tạo dataset mẫu tại {LOCAL_DATA_DIR}')

### Bước 6: Huấn luyện chính thức trên GPU Colab (Lưu `best_model.pth` trực tiếp vào Google Drive)
- Để bật thêm 2 kênh giải tích tường minh **Donoho MAD + ELA (11 kênh)**, thêm cờ `--use-forensic-maps`.
- Để tiếp tục huấn luyện khi Colab bị ngắt kết nối, thêm cờ `--resume {DRIVE_CKPT_DIR}/last_model.pth`.

In [ ]:
!python train.py \
    --data_dir {LOCAL_DATA_DIR} \
    --save_dir {DRIVE_CKPT_DIR} \
    --epochs 50 \
    --batch_size 16 \
    --lr 0.001 \
    --amp

### Bước 7: Xuất ONNX & Lượng tử hóa INT8 (`< 2 MB`) cho Raspberry Pi 3
Chuyển đổi `best_model.pth` sang `model_fp32.onnx` và lượng tử hóa INT8 (`model_int8.onnx`), lưu trực tiếp vào Google Drive và thư mục `checkpoints/`.

In [ ]:
!python export_quantize.py \
    --checkpoint {DRIVE_CKPT_DIR}/best_model.pth \
    --output_dir {DRIVE_CKPT_DIR}

# Copy model INT8 về thư mục checkpoints/ của repo để chạy demo hoặc commit/download về Raspberry Pi 3
!cp -f {DRIVE_CKPT_DIR}/model_int8.onnx ./checkpoints/model_int8.onnx
!cp -f {DRIVE_CKPT_DIR}/best_model.pth ./checkpoints/best_model.pth
!ls -lh {DRIVE_CKPT_DIR}

### Bước 8: Khởi chạy Giao diện Demo Gradio trực tiếp trên Google Colab (`share=True`)

In [ ]:
!python app_colab.py --model_path {DRIVE_CKPT_DIR}/model_int8.onnx --share